In [ ]:
# Modules Load/Reload
# ── CONFIG ────────────────────────────────────────────────────────────────────
DEV_MODE = True   # True: reload modules on every run | False: simple import
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

if DEV_MODE:
    import importlib
    import ticker_provider  as _tp; importlib.reload(_tp)
    import options_scanner  as _os; importlib.reload(_os)
    import options_archive  as _oa; importlib.reload(_oa)

from ticker_provider import get_tickers
from options_scanner  import OptionsScanner

print(f"Modules {'reloaded' if DEV_MODE else 'imported'} | DEV_MODE={'ON' if DEV_MODE else 'OFF'}")


In [ ]:
# CONFIG
# Available universes (see ticker_provider.py SOURCES):
# US:     sp500, nasdaq100, dow30, russell2000
# Europe: dax40, cac40, ftse100, eurostoxx50, aex25, ftse_mib, ibex35
UNIVERSE = "sp500"

# ── Ticker list cache ─────────────────────────────────────────────────────────
TICKERS_CACHE_HOURS = 168   # refresh the list weekly (Wikipedia scrape)
CACHE_FOLDER         = "cache"

# ── Options scanner ───────────────────────────────────────────────────────────
MIN_DAYS_TO_EXPIRY = 7
MAX_DAYS_TO_EXPIRY = 60
SLEEP_BETWEEN_TICKERS = 0.3   # seconds — yfinance rate limiting

# ── Archiving ──────────────────────────────────────────────────────────────
# Relative folder — no dependency on any shared_data_layer path.
# On GitHub Actions this stays inside the runner workspace; download it
# via upload-artifact (see workflow yaml) before the job closes.
ARCHIVE_FOLDER = "options_archive"


In [ ]:
# Ticker List (today's list, NOT point-in-time — not needed here)
tickers = get_tickers(UNIVERSE, folder_path=CACHE_FOLDER, max_age_hours=TICKERS_CACHE_HOURS)
print(f"Universe: {UNIVERSE} -> {len(tickers)} tickers (current composition)")


In [ ]:
# Options Scan + Archive
# No scoring here — just download + save the raw chain (strike-level).
# The MR script reads from the options_archive/ folder later.
scanner = OptionsScanner(
    min_days = MIN_DAYS_TO_EXPIRY,
    max_days = MAX_DAYS_TO_EXPIRY,
    sleep    = SLEEP_BETWEEN_TICKERS,
)
scanner.scan(tickers)

saved_path = scanner.save_full_chain_archive(folder_path=ARCHIVE_FOLDER)
if saved_path:
    print(f"Saved: {saved_path}")

scanner.print_report(min_pcr=1.0, top_n=25)


## Note

This notebook produces **only** the raw options chain archive
(`options_archive/options_YYYY-MM-DD.parquet`). The per-ticker aggregate
DataFrame (PCR/IV summary, no strike-level detail) is available via
`scanner.to_dataframe()` for a quick sanity check — but what the MR script
actually consumes downstream is the archived Parquet file, not anything
held in-memory in this notebook (each run/machine is independent).
